# NASDAQ-100: Is Today's Market the Dot-Com Bubble Again?
### A P/E & P/B Ratio Comparison — 1999–2001 vs 2025–2026

**Author analysis notebook** | Data: `PEPB.xlsx` (5 sheets: `1999`, `2000`, `2001`, `2025`, `2026`)

**Valuation snapshot dates**
| Period | Snapshot date |
|---|---|
| 1999 / 2000 / 2001 (Dot-Com Era) | 31 March of that year |
| 2025 / 2026 (Current Era) | 20 June of that year |

> Note: because the two eras are measured on different calendar days (31-Mar vs 20-Jun), this notebook treats each year as a **standalone annual snapshot** rather than a continuously aligned time series. All comparisons are "snapshot vs snapshot," which is the right way to read cross-era valuation comparisons anyway — we care about the regime each snapshot represents, not the exact day count between them.

**What this notebook does**
1. Loads and cleans all 5 sheets
2. Plots P/E and P/B for every firm, every year — split into 2 readable halves per year (**10 combined figures → 20 panels**, as requested)
3. Computes descriptive statistics (mean, median, std, quartiles, dispersion) for every year
4. Goes a bit further: a median-trend chart across the two eras, a survivor-company case study (firms present in both 1999–2001 and 2025–2026), and a P/E-vs-P/B "valuation landscape" scatter
5. Closes with a direct, evidence-based answer to the dot-com question


## 1. Setup & Imports

In [1]:
import pandas as pd
import numpy as np
import re
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 30)

# Plotly theme used throughout the notebook
TEMPLATE = 'plotly_dark'


## 2. Load the Workbook

The file has 5 sheets — three from the dot-com era (`1999`, `2000`, `2001`, each priced as of **31 March**) and two from today (`2025`, `2026`, each priced as of **20 June**). Each sheet has the same 3 columns: `Company Name`, `P/E`, `P/B`.

In [2]:
SRC = 'PEPB.xlsx'  # place this notebook in the same folder as PEPB.xlsx, or update the path

xl = pd.ExcelFile(SRC)
YEARS = ['1999', '2000', '2001', '2025', '2026']

ERA_MAP  = {'1999': 'Dot-Com Era', '2000': 'Dot-Com Era', '2001': 'Dot-Com Era',
            '2025': 'Current Era', '2026': 'Current Era'}
DATE_MAP = {'1999': '31-Mar-1999', '2000': '31-Mar-2000', '2001': '31-Mar-2001',
            '2025': '20-Jun-2025', '2026': '20-Jun-2026'}
ERA_COLOR = {'Dot-Com Era': '#EF553B', 'Current Era': '#00CC96'}

data = {}
for y in YEARS:
    df = xl.parse(y).copy()
    df['Year'] = y
    df['Era'] = ERA_MAP[y]
    df['AsOfDate'] = DATE_MAP[y]
    data[y] = df

for y in YEARS:
    print(f"{y} ({ERA_MAP[y]}, as of {DATE_MAP[y]}): {data[y].shape[0]} firms, "
          f"{data[y]['P/E'].isna().sum()} with no P/E (negative/undefined earnings), "
          f"{data[y]['P/B'].isna().sum()} with no P/B")


1999 (Dot-Com Era, as of 31-Mar-1999): 45 firms, 9 with no P/E (negative/undefined earnings), 2 with no P/B
2000 (Dot-Com Era, as of 31-Mar-2000): 45 firms, 5 with no P/E (negative/undefined earnings), 1 with no P/B
2001 (Dot-Com Era, as of 31-Mar-2001): 45 firms, 10 with no P/E (negative/undefined earnings), 5 with no P/B
2025 (Current Era, as of 20-Jun-2025): 50 firms, 7 with no P/E (negative/undefined earnings), 4 with no P/B
2026 (Current Era, as of 20-Jun-2026): 50 firms, 2 with no P/E (negative/undefined earnings), 3 with no P/B


In [3]:
# Quick peek
data['1999'].head()


,Company Name,P/E,P/B,Year,Era,AsOfDate
0,Microsoft Corporation,26.832335,6.658247,1999,Dot-Com Era,31-Mar-1999
1,Intel Corporation,141.953757,34.834043,1999,Dot-Com Era,31-Mar-1999
2,"Cisco Systems, Inc.",34.780488,4.961726,1999,Dot-Com Era,31-Mar-1999
3,"MCI WorldCom, Inc.",NaN,3.542986,1999,Dot-Com Era,31-Mar-1999
4,Dell Computer Corporation,261.312500,37.666667,1999,Dot-Com Era,31-Mar-1999


## 3. Cleaning Notes

- Missing `P/E` values are firms with **negative or undefined earnings** that quarter (P/E is not meaningful for a loss-making company) — these are left as `NaN` rather than dropped or zero-filled, and are excluded from ratio statistics but **kept in the company list** so the bar charts still show them as gaps.
- Company names are shortened for chart readability (e.g. *"Microsoft Corporation"* → *"Microsoft"*) using a small regex helper — the **full legal name is preserved** in the underlying dataframe and tooltips reference the original where useful.
- No duplicate company names were found in any sheet.
- Each year's firms are kept in their **original sheet order**, which follows index-weight rank (largest first). This order is preserved when splitting each year into two halves, so "Part 1" is consistently the larger/more heavily-weighted names and "Part 2" is the smaller ones — making the two-part split itself analytically meaningful rather than arbitrary.


In [4]:
SUFFIX_RE = re.compile(r',?\s*(Incorporated|Corporation|Inc\.|Co\.|plc|Holdings|Ltd\.|PLC|Company)\.?\s*$')

def shorten(name: str) -> str:
    # Strip common corporate suffixes for cleaner axis labels.
    out = SUFFIX_RE.sub('', name).strip()
    return out if out else name

def split_half(df: pd.DataFrame):
    # Split a year's firms into two roughly-equal halves, preserving original (index-weight) order.
    half = int(np.ceil(len(df) / 2))
    return df.iloc[:half].copy(), df.iloc[half:].copy()


## 4. P/E & P/B by Firm — Every Year, Split Into 2 Parts

Per your spec: each of the 5 years is split into **2 parts** (since there are too many firms to read on one axis), each part is plotted for **both P/E and P/B**.

5 years × 2 parts × 2 metrics = **20 chart panels**, delivered as **10 combined Plotly figures** (one figure per year-part, with P/E and P/B side by side so they share the same company axis and are easy to compare directly).

Both axes use a **log scale** — dot-com era P/E ratios swing from under 1x to over 4,500x in the same chart, and a linear axis would make everything except the single biggest outlier unreadable.


In [5]:
def make_year_part_fig(df_part: pd.DataFrame, year: str, part_label: str):
    df_plot = df_part.copy()
    df_plot['ShortName'] = df_plot['Company Name'].apply(shorten)
    color = ERA_COLOR[ERA_MAP[year]]

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=("P/E Ratio (log scale)", "P/B Ratio (log scale)"),
        horizontal_spacing=0.10
    )
    fig.add_trace(go.Bar(
        x=df_plot['ShortName'], y=df_plot['P/E'], marker_color=color, showlegend=False,
        hovertemplate='<b>%{x}</b><br>P/E: %{y:.1f}x<extra></extra>'
    ), row=1, col=1)
    fig.add_trace(go.Bar(
        x=df_plot['ShortName'], y=df_plot['P/B'], marker_color=color, opacity=0.8, showlegend=False,
        hovertemplate='<b>%{x}</b><br>P/B: %{y:.1f}x<extra></extra>'
    ), row=1, col=2)

    fig.update_yaxes(type='log', title_text='P/E (x)', row=1, col=1)
    fig.update_yaxes(type='log', title_text='P/B (x)', row=1, col=2)
    fig.update_xaxes(tickangle=-45, row=1, col=1)
    fig.update_xaxes(tickangle=-45, row=1, col=2)
    fig.update_layout(
        title=f"NASDAQ-100 Top Firms — {year} ({ERA_MAP[year]}) | {part_label}"
              f"<br><sup>Valuation as of {DATE_MAP[year]} · bars with no value = negative/undefined earnings (P/E) or book value (P/B)</sup>",
        template=TEMPLATE, height=520, width=1150, margin=dict(t=110)
    )
    return fig

# Build and store all 10 figures (5 years x 2 parts), keyed for later reference
year_part_figs = {}
for y in YEARS:
    p1, p2 = split_half(data[y])
    year_part_figs[(y, 1)] = make_year_part_fig(p1, y, "Part 1 of 2 — larger index weights")
    year_part_figs[(y, 2)] = make_year_part_fig(p2, y, "Part 2 of 2 — smaller index weights")

print(f"Built {len(year_part_figs)} figures (5 years x 2 parts) covering 20 chart panels total.")


Built 10 figures (5 years x 2 parts) covering 20 chart panels total.


### 4.1 — 1999 (Dot-Com Era, as of 31-Mar-1999)

In [6]:
year_part_figs[('1999', 1)].show()

In [7]:
year_part_figs[('1999', 2)].show()

### 4.2 — 2000 (Dot-Com Era, as of 31-Mar-2000)

In [8]:
year_part_figs[('2000', 1)].show()

In [9]:
year_part_figs[('2000', 2)].show()

### 4.3 — 2001 (Dot-Com Era, as of 31-Mar-2001)

In [10]:
year_part_figs[('2001', 1)].show()

In [11]:
year_part_figs[('2001', 2)].show()

### 4.4 — 2025 (Current Era, as of 20-Jun-2025)

In [12]:
year_part_figs[('2025', 1)].show()

In [13]:
year_part_figs[('2025', 2)].show()

### 4.5 — 2026 (Current Era, as of 20-Jun-2026)

In [14]:
year_part_figs[('2026', 1)].show()

In [15]:
year_part_figs[('2026', 2)].show()

## 5. Descriptive Statistics by Year

For each year: count of firms, how many have a meaningful P/E (i.e. positive earnings), mean/median/std, min/max, quartiles, and two "bubble-watch" indicators —
the **% of firms trading above 100x earnings** and the **% trading above 10x book value** — both classic stretched-valuation thresholds.


In [16]:
def describe_year(df: pd.DataFrame, y: str) -> dict:
    pe = df['P/E'].dropna()
    pb = df['P/B'].dropna()
    return {
        'Year': y, 'Era': ERA_MAP[y], 'As Of': DATE_MAP[y],
        'N Firms': len(df),
        'N Valid P/E': len(pe),
        'N P/E N/A (neg. earnings)': int(df['P/E'].isna().sum()),
        'P/E Mean': pe.mean(), 'P/E Median': pe.median(), 'P/E Std Dev': pe.std(),
        'P/E Min': pe.min(), 'P/E Max': pe.max(),
        'P/E Q1': pe.quantile(.25), 'P/E Q3': pe.quantile(.75),
        'P/B Mean': pb.mean(), 'P/B Median': pb.median(), 'P/B Std Dev': pb.std(),
        'P/B Min': pb.min(), 'P/B Max': pb.max(),
        'P/B Q1': pb.quantile(.25), 'P/B Q3': pb.quantile(.75),
        '% Firms P/E > 100x': round((pe > 100).mean() * 100, 1),
        '% Firms P/B > 10x': round((pb > 10).mean() * 100, 1),
    }

summary_df = pd.DataFrame([describe_year(data[y], y) for y in YEARS]).set_index('Year')
summary_df.round(2).T


Year,1999,2000,2001,2025,2026
Era,Dot-Com Era,Dot-Com Era,Dot-Com Era,Current Era,Current Era
As Of,31-Mar-1999,31-Mar-2000,31-Mar-2001,20-Jun-2025,20-Jun-2026
N Firms,45,45,45,50,50
N Valid P/E,36,40,35,43,48
N P/E N/A (neg. earnings),9,5,10,7,2
P/E Mean,256.51,250.52,136.12,53.26,73.29
P/E Median,89.68,93.51,57.95,30.22,34.54
P/E Std Dev,761.45,429.1,204.05,58.83,117.42
P/E Min,0.34,1.17,8.03,1.4,4.42
P/E Max,4525.0,2175.0,1028.46,283.33,571.79


**Reading the table:**
- **Median P/E** is the single most useful "apples-to-apples" number here, since the mean is heavily distorted by a handful of extreme outliers in every year (note how much higher Mean is than Median in 1999/2000 — that gap *is* the bubble signature).
- **% of firms above 100x P/E** is a blunt but effective bubble gauge: in the dot-com years, 34–48% of these top NASDAQ firms traded above 100x earnings. Watch what that number does in 2025/2026.


In [17]:
fig_bar_stats = go.Figure()
fig_bar_stats.add_trace(go.Bar(x=summary_df.index, y=summary_df['P/E Median'],
                                name='Median P/E', marker_color=[ERA_COLOR[ERA_MAP[y]] for y in YEARS],
                                text=summary_df['P/E Median'].round(1), textposition='outside'))
fig_bar_stats.update_layout(title='Median P/E by Year — the cleanest single bubble gauge',
                             template=TEMPLATE, height=450, width=800,
                             yaxis_title='Median P/E (x)', showlegend=False)
fig_bar_stats.show()


In [18]:
fig_pct100 = go.Figure()
fig_pct100.add_trace(go.Bar(x=summary_df.index, y=summary_df['% Firms P/E > 100x'],
                             marker_color=[ERA_COLOR[ERA_MAP[y]] for y in YEARS],
                             text=summary_df['% Firms P/E > 100x'].astype(str) + '%', textposition='outside'))
fig_pct100.update_layout(title='Share of Firms Trading Above 100x Earnings',
                          template=TEMPLATE, height=450, width=800,
                          yaxis_title='% of firms', showlegend=False)
fig_pct100.show()


## 6. Bonus Analysis #1 — Median P/E & P/B Across Both Eras

A single chart connecting all 5 snapshots, with P/E and P/B on separate axes since they live on very different scales.


In [19]:
trend_fig = make_subplots(specs=[[{"secondary_y": True}]])
trend_fig.add_trace(go.Scatter(x=summary_df.index, y=summary_df['P/E Median'], mode='lines+markers',
                                name='Median P/E', line=dict(color='#EF553B', width=3),
                                marker=dict(size=10)), secondary_y=False)
trend_fig.add_trace(go.Scatter(x=summary_df.index, y=summary_df['P/B Median'], mode='lines+markers',
                                name='Median P/B', line=dict(color='#636EFA', width=3, dash='dot'),
                                marker=dict(size=10)), secondary_y=True)
trend_fig.add_vrect(x0=1.5, x1=2.5, fillcolor='gray', opacity=0.12, line_width=0,
                     annotation_text='gap = no data\n(2002-2024 not in dataset)', annotation_position='top')
trend_fig.update_layout(title='Median P/E & P/B: Dot-Com Era (1999-2001) vs Current Era (2025-2026)',
                         template=TEMPLATE, height=480, width=950,
                         legend=dict(orientation='h', y=1.1))
trend_fig.update_yaxes(title_text='Median P/E (x)', secondary_y=False)
trend_fig.update_yaxes(title_text='Median P/B (x)', secondary_y=True)
trend_fig.show()


Note the gap between 2001 and 2025 is **not** a smooth multi-year trend — there's a 24-year hole in this dataset. Treat each side of the gray band as its own 3-year (or 2-year) story, not as one continuous line.


## 7. Bonus Analysis #2 — The Survivors: Same Companies, Both Eras

A handful of companies appear in **both** the dot-com snapshots and today's. Tracking the *same business* through both regimes is a cleaner comparison than comparing different sets of companies, because it removes "different company mix" as a confounder.


In [20]:
old_set = set(data['1999']['Company Name']) | set(data['2000']['Company Name']) | set(data['2001']['Company Name'])
new_set = set(data['2025']['Company Name']) | set(data['2026']['Company Name'])
survivors = sorted(old_set & new_set)
print(f"{len(survivors)} companies appear in both the dot-com snapshots and today's snapshots:")
for s in survivors:
    print(" -", s)


8 companies appear in both the dot-com snapshots and today's snapshots:
 - Amazon.com, Inc.
 - Amgen Inc.
 - Applied Materials, Inc.
 - Cisco Systems, Inc.
 - Comcast Corporation
 - Intel Corporation
 - QUALCOMM Incorporated
 - Starbucks Corporation


In [21]:
surv_rows = []
for y in YEARS:
    sub = data[y][data[y]['Company Name'].isin(survivors)]
    for _, r in sub.iterrows():
        surv_rows.append({'Year': y, 'Company': shorten(r['Company Name']), 'P/E': r['P/E'], 'P/B': r['P/B']})
surv_df = pd.DataFrame(surv_rows)

surv_pivot_pe = surv_df.pivot(index='Company', columns='Year', values='P/E')
surv_pivot_pe


Year,1999,2000,2001,2025,2026
Company,,,,,
Amazon.com,NaN,NaN,NaN,39.8650,29.2333
Amgen,132.913043,224.000000,109.714286,30.1700,23.4934
Applied Materials,212.285714,122.166667,97.444444,20.6911,57.9446
Cisco Systems,34.780488,144.437500,41.414634,27.9831,38.8117
Comcast,16.666667,49.746154,9.713615,8.5163,4.4154
Intel,141.953757,54.796209,57.947020,NaN,NaN
QUALCOMM,193.573529,536.000000,58.666667,16.2885,24.5772
Starbucks,101.625000,69.206897,57.285714,35.2727,76.6617


In [22]:
surv_fig_pe = px.line(surv_df, x='Year', y='P/E', color='Company', markers=True, log_y=True,
                       title='Survivors\u2019 P/E Trajectory: 1999\u20132001 vs 2025\u20132026')
surv_fig_pe.update_layout(template=TEMPLATE, height=500, width=1000)
surv_fig_pe.show()


In [23]:
surv_fig_pb = px.line(surv_df, x='Year', y='P/B', color='Company', markers=True, log_y=True,
                       title='Survivors\u2019 P/B Trajectory: 1999\u20132001 vs 2025\u20132026')
surv_fig_pb.update_layout(template=TEMPLATE, height=500, width=1000)
surv_fig_pb.show()


**What this shows:** several survivors (e.g. Cisco, Intel, Qualcomm) carried much richer earnings multiples in 1999–2000 than their 2025–2026 selves do today — consistent with the broader median-P/E finding above. Amazon has no P/E bar in the dot-com years at all, because it had negative earnings then; today it's profitable. That's a useful reminder that "no P/E" in 1999–2001 usually meant *unprofitable growth story*, which was itself a hallmark of the bubble.


## 8. Bonus Analysis #3 — The P/E vs P/B "Valuation Landscape"

Plotting every firm-year on one P/E–P/B scatter (log-log) shows where each era's cloud of points sits relative to the other.


In [24]:
pooled = pd.concat([data[y] for y in YEARS], ignore_index=True).dropna(subset=['P/E', 'P/B'])
pooled['ShortName'] = pooled['Company Name'].apply(shorten)

scatter_fig = px.scatter(pooled, x='P/E', y='P/B', color='Era', symbol='Year',
                          hover_name='ShortName', log_x=True, log_y=True,
                          color_discrete_map=ERA_COLOR, opacity=0.75,
                          title='P/E vs P/B Landscape \u2014 Dot-Com Era vs Current Era (log\u2013log)')
scatter_fig.update_traces(marker=dict(size=11, line=dict(width=0.5, color='white')))
scatter_fig.update_layout(template=TEMPLATE, height=560, width=1000,
                           xaxis_title='P/E (log scale)', yaxis_title='P/B (log scale)')
scatter_fig.show()


**How to read this:** points further right = richer earnings multiple; points further up = richer book-value multiple. If today's market were a simple replay of the dot-com bubble, the green (Current Era) cloud would sit on top of the red (Dot-Com Era) cloud. It doesn't — the comparison is more nuanced than that, as the next section spells out.


## 9. So — Is This the Dot-Com Bubble Again?

**Short answer: not quite the same shape, but not entirely different either. It's a different kind of expensive.**

**Where today looks *less* extreme than 1999–2001 (on earnings):**
- Median P/E in the dot-com years was **~58–94x**; today it's **~30–35x** — roughly a third of the dot-com multiple.
- The share of firms trading above 100x earnings was **34–48%** in 1999–2001; today it's only **~12%**.
- Today's earnings multiples are also driven by a *smaller* set of extreme outliers — the mean-to-median gap, while still present (thanks to a few very expensive names), is proportionally tighter than in 1999–2000.

**Where today looks *just as rich*, or richer (on book value):**
- Median P/B today (**~8.6–10.5x**) is comparable to, or higher than, the dot-com median (**~4.2–8.0x**).
- The share of firms trading above 10x book value is actually **higher today (43–53%) than in any dot-com year (21–41%)**.

**The honest takeaway:** the 1999–2001 NASDAQ-100 was expensive *on both earnings and book value at once* — classic broad-based bubble pricing, with many unprofitable "story stocks" carrying no P/E at all. Today's NASDAQ-100 is **cheaper relative to earnings** (today's mega-caps are, for the most part, genuinely very profitable) but **just as rich, or richer, relative to book value** — consistent with a market paying up for a smaller set of dominant, high-margin franchises rather than speculating broadly on unprofitable growth.

So: this is **not** a carbon-copy of the dot-com bubble. It's a narrower, earnings-backed version of "expensive" — which doesn't rule out a correction, but it is a meaningfully different risk profile than 1999–2001, where the danger was broad-based earnings-free speculation.

> *This is a descriptive read of the two P/E and P/B snapshots provided — it isn't investment advice, and a full "bubble or not" judgment would also want to look at things this dataset doesn't contain: earnings growth rates, interest-rate regime, market breadth beyond the top ~50 names, and IPO/speculative-financing activity.*
